In [1]:
import re
from collections import defaultdict
from typing import List, Tuple, Dict, Set

In [2]:
class RabinKarpDetector:
    """
    A comprehensive system for plagiarism detection and keyword monitoring
    using the Rabin-Karp algorithm.
    """
    
    def __init__(self, d=256, q=101):
        """
        Initialize the Rabin-Karp detector.
        
        Args:
            d: Base for hashing (default 256)
            q: Prime number for modulo operation (default 101)
        """
        self.d = d
        self.q = q
    
    def _compute_hash(self, text: str, length: int) -> int:
        """Compute hash value for a text segment."""
        hash_val = 0
        for i in range(length):
            hash_val = (self.d * hash_val + ord(text[i])) % self.q
        return hash_val
    
    def _rolling_hash(self, old_hash: int, old_char: str, new_char: str, h: int) -> int:
        """Update hash value using rolling hash technique."""
        new_hash = (self.d * (old_hash - ord(old_char) * h) + ord(new_char)) % self.q
        return new_hash if new_hash >= 0 else new_hash + self.q
    
    def tokenize_text(self, text: str) -> List[str]:
        """
        Tokenize text into words, removing punctuation and converting to lowercase.
        """
        # Remove punctuation and convert to lowercase
        text = re.sub(r'[^\w\s]', ' ', text.lower())
        # Split into words and filter out empty strings
        words = [word for word in text.split() if word]
        return words
    
    def detect_plagiarism(self, document1: str, document2: str, 
                         window_size: int = 5, 
                         threshold: float = 0.3) -> Dict:
        """
        Detect plagiarism between two documents by finding matching word sequences.
        
        Args:
            document1: First document text
            document2: Second document text
            window_size: Number of consecutive words to compare (n-gram size)
            threshold: Similarity threshold (0-1) to flag as plagiarism
        
        Returns:
            Dictionary containing similarity score and matching segments
        """
        # Tokenize documents
        words1 = self.tokenize_text(document1)
        words2 = self.tokenize_text(document2)
        
        if len(words1) < window_size or len(words2) < window_size:
            return {
                'similarity_score': 0.0,
                'is_plagiarism': False,
                'matching_segments': [],
                'details': 'Documents too short for comparison'
            }
        
        # Create n-grams and their hashes for document 2
        doc2_hashes = defaultdict(list)
        
        for i in range(len(words2) - window_size + 1):
            ngram = ' '.join(words2[i:i + window_size])
            hash_val = self._compute_hash(ngram, len(ngram))
            doc2_hashes[hash_val].append((i, ngram))
        
        # Find matches in document 1
        matches = []
        matched_positions = set()
        
        for i in range(len(words1) - window_size + 1):
            ngram = ' '.join(words1[i:i + window_size])
            hash_val = self._compute_hash(ngram, len(ngram))
            
            # Check if hash exists in document 2
            if hash_val in doc2_hashes:
                for doc2_pos, doc2_ngram in doc2_hashes[hash_val]:
                    # Verify actual match (handle hash collisions)
                    if ngram == doc2_ngram and i not in matched_positions:
                        matches.append({
                            'doc1_position': i,
                            'doc2_position': doc2_pos,
                            'matched_text': ngram,
                            'word_count': window_size
                        })
                        matched_positions.add(i)
                        break
        
        # Calculate similarity score
        total_ngrams = len(words1) - window_size + 1
        similarity_score = len(matches) / total_ngrams if total_ngrams > 0 else 0
        
        return {
            'similarity_score': round(similarity_score, 4),
            'is_plagiarism': similarity_score >= threshold,
            'matching_segments': matches,
            'total_matches': len(matches),
            'doc1_word_count': len(words1),
            'doc2_word_count': len(words2)
        }
    
    def monitor_keywords(self, text: str, keywords: List[str], 
                        case_sensitive: bool = False) -> Dict:
        """
        Monitor text for presence of specific keywords or phrases.
        Useful for content moderation and threat detection.
        
        Args:
            text: Text to monitor
            keywords: List of keywords/phrases to search for
            case_sensitive: Whether search should be case-sensitive
        
        Returns:
            Dictionary with detected keywords and their positions
        """
        if not case_sensitive:
            text = text.lower()
            keywords = [kw.lower() for kw in keywords]
        
        detections = defaultdict(list)
        
        for keyword in keywords:
            n = len(text)
            m = len(keyword)
            
            if m > n:
                continue
            
            # Calculate hash for pattern
            p = self._compute_hash(keyword, m)
            t = self._compute_hash(text[:m], m)
            
            # Calculate h = d^(m-1) % q
            h = 1
            for i in range(m - 1):
                h = (h * self.d) % self.q
            
            # Slide pattern over text
            for i in range(n - m + 1):
                if p == t:
                    # Verify match
                    if text[i:i + m] == keyword:
                        # Find context (30 chars before and after)
                        start = max(0, i - 30)
                        end = min(n, i + m + 30)
                        context = text[start:end]
                        
                        detections[keyword].append({
                            'position': i,
                            'context': context
                        })
                
                # Calculate hash for next window
                if i < n - m:
                    t = self._rolling_hash(t, text[i], text[i + m], h)
        
        return {
            'detected_keywords': list(detections.keys()),
            'total_detections': sum(len(v) for v in detections.values()),
            'detections': dict(detections),
            'is_flagged': len(detections) > 0
        }
    
    def compare_multiple_documents(self, documents: Dict[str, str], 
                                  window_size: int = 5,
                                  threshold: float = 0.3) -> List[Dict]:
        """
        Compare multiple documents for plagiarism detection.
        
        Args:
            documents: Dictionary with document names as keys and text as values
            window_size: N-gram size for comparison
            threshold: Similarity threshold
        
        Returns:
            List of plagiarism matches between document pairs
        """
        doc_names = list(documents.keys())
        results = []
        
        for i in range(len(doc_names)):
            for j in range(i + 1, len(doc_names)):
                doc1_name = doc_names[i]
                doc2_name = doc_names[j]
                
                comparison = self.detect_plagiarism(
                    documents[doc1_name],
                    documents[doc2_name],
                    window_size,
                    threshold
                )
                
                if comparison['is_plagiarism']:
                    results.append({
                        'document1': doc1_name,
                        'document2': doc2_name,
                        'similarity_score': comparison['similarity_score'],
                        'matches': comparison['total_matches']
                    })
        
        return sorted(results, key=lambda x: x['similarity_score'], reverse=True)

In [3]:
# Example Usage and Demonstrations
if __name__ == "__main__":
    detector = RabinKarpDetector()
    
    print("=" * 70)
    print("PLAGIARISM DETECTION DEMO")
    print("=" * 70)
    
    # Example documents
    essay1 = """
    Climate change represents one of the most significant challenges facing 
    humanity today. The rising global temperatures are causing ice caps to melt 
    and sea levels to rise. Scientists agree that human activities are the 
    primary cause of recent climate change. We must take action now to reduce 
    carbon emissions and protect our planet for future generations.
    """
    
    essay2 = """
    The rising global temperatures are causing ice caps to melt and sea levels 
    to rise dramatically. This phenomenon represents a major threat to coastal 
    cities. Scientists agree that human activities are the primary cause of 
    recent climate change and we need immediate action. Renewable energy 
    sources offer hope for a sustainable future.
    """
    
    result = detector.detect_plagiarism(essay1, essay2, window_size=5, threshold=0.2)
    
    print(f"\nSimilarity Score: {result['similarity_score']:.2%}")
    print(f"Plagiarism Detected: {result['is_plagiarism']}")
    print(f"Total Matches Found: {result['total_matches']}")
    
    if result['matching_segments']:
        print(f"\nMatching Segments (showing first 3):")
        for match in result['matching_segments'][:3]:
            print(f"  - '{match['matched_text']}'")
    
    print("\n" + "=" * 70)
    print("KEYWORD MONITORING DEMO (Content Moderation)")
    print("=" * 70)
    
    # Example chat messages
    messages = [
        "Hey everyone! Great discussion today.",
        "I need help with my homework on algorithms.",
        "This is spam content buy now click here!!!",
        "Let me share my credit card number 1234-5678-9012-3456"
    ]
    
    # Keywords to flag
    flagged_keywords = ["spam", "buy now", "click here", "credit card"]
    
    for i, message in enumerate(messages):
        result = detector.monitor_keywords(message, flagged_keywords)
        print(f"\nMessage {i + 1}: \"{message}\"")
        print(f"Flagged: {result['is_flagged']}")
        if result['is_flagged']:
            print(f"Detected: {', '.join(result['detected_keywords'])}")
    
    print("\n" + "=" * 70)
    print("MULTIPLE DOCUMENT COMPARISON")
    print("=" * 70)
    
    documents = {
        "Student A": "The quick brown fox jumps over the lazy dog.",
        "Student B": "A fast brown fox leaps over a sleepy dog.",
        "Student C": "The quick brown fox jumps over the lazy dog enthusiastically."
    }
    
    comparisons = detector.compare_multiple_documents(documents, window_size=3, threshold=0.3)
    
    if comparisons:
        print("\nPlagiarism detected between:")
        for comp in comparisons:
            print(f"  {comp['document1']} ↔ {comp['document2']}: "
                  f"{comp['similarity_score']:.2%} similar")
    else:
        print("\nNo significant plagiarism detected.")

PLAGIARISM DETECTION DEMO

Similarity Score: 38.46%
Plagiarism Detected: True
Total Matches Found: 20

Matching Segments (showing first 3):
  - 'the rising global temperatures are'
  - 'rising global temperatures are causing'
  - 'global temperatures are causing ice'

KEYWORD MONITORING DEMO (Content Moderation)

Message 1: "Hey everyone! Great discussion today."
Flagged: False

Message 2: "I need help with my homework on algorithms."
Flagged: False

Message 3: "This is spam content buy now click here!!!"
Flagged: True
Detected: spam, buy now, click here

Message 4: "Let me share my credit card number 1234-5678-9012-3456"
Flagged: True
Detected: credit card

MULTIPLE DOCUMENT COMPARISON

Plagiarism detected between:
  Student A ↔ Student C: 100.00% similar
